# Формирование хронологической выборки

## Импортирование библиотек и настройка параметров

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

import joblib

## Настройка параметров

In [ ]:
PROCESSED_PATH = Path("../data/processed/cpi_monthly.csv")
OUTPUT_DIR = Path("../data/processed")

WINDOW_SIZE = 12
SPLIT_DATE = pd.Timestamp('2022-01-01')

## Загрузка исходных данных

In [ ]:
df = pd.read_csv(PROCESSED_PATH, parse_dates=["Date"])
print(f"Исходный ряд: {df.shape[0]} наблюдений")

## Хронологическое разбиение и масштабирование

In [ ]:
train_raw = df[df['Date'] < SPLIT_DATE].copy()
test_raw = df[df['Date'] >= SPLIT_DATE].copy()

print(f"Train: {len(train_raw)} наблюдений ({train_raw['Date'].min().date()} — {train_raw['Date'].max().date()})")
print(f"Test:  {len(test_raw)} наблюдений ({test_raw['Date'].min().date()} — {test_raw['Date'].max().date()})")

scaler_r2 = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler_r2.fit_transform(train_raw['cpi_mom'].values.reshape(-1, 1))
test_scaled = scaler_r2.transform(test_raw['cpi_mom'].values.reshape(-1, 1))

## Генерация скользящих окон

In [ ]:
def create_sequences(data, window_size):
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i-window_size:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_train_r2, y_train_r2 = create_sequences(train_scaled, WINDOW_SIZE)

test_with_history = np.concatenate([train_scaled[-WINDOW_SIZE:], test_scaled])
X_test_r2, y_test_r2 = create_sequences(test_with_history, WINDOW_SIZE)

X_train_r2 = np.reshape(X_train_r2, (X_train_r2.shape[0], X_train_r2.shape[1], 1))
X_test_r2 = np.reshape(X_test_r2, (X_test_r2.shape[0], X_test_r2.shape[1], 1))

print(f"X_train R2: {X_train_r2.shape}")
print(f"y_train R2: {y_train_r2.shape}")
print(f"X_test R2:  {X_test_r2.shape}")
print(f"y_test R2:  {y_test_r2.shape}")

## Сохранение данных

In [ ]:
np.save(OUTPUT_DIR / "X_train_r2.npy", X_train_r2)
np.save(OUTPUT_DIR / "y_train_r2.npy", y_train_r2)
np.save(OUTPUT_DIR / "X_test_r2.npy", X_test_r2)
np.save(OUTPUT_DIR / "y_test_r2.npy", y_test_r2)
joblib.dump(scaler_r2, OUTPUT_DIR / "scaler_r2.joblib")

print("Файлы R2 успешно сохранены.")